A notebook for processing the log files into a format suitable for input to the LLM evaluation.

In [ ]:
import json
import os
import shutil

In [ ]:
SESSIONS_PATH = "sessions/..."
OUTPUT_PATH = "sessions/..."
NAIVE = False

In [ ]:
outputs = []
for filename in sorted(
    [name for name in os.listdir(SESSIONS_PATH) if name != ".DS_Store"]
):
    print(filename)
    json_ = json.load(open(SESSIONS_PATH + "/" + filename, "r"))
    if not NAIVE:
        for k, v in json_.items():
            if v["function_ran"] == "_run_writing_agent":
                final_response = v
                break
        writer_input = final_response["kwargs"].split("'data': ")[-1].rstrip("}")
        writer_response = final_response["res"]
        runtime = json_[list(json_.keys())[-1]]["function_runtime"]
        query = json_["0"]["kwargs"][1:-1].split(": ")[-1].strip("'").strip('"')
    elif NAIVE:
        writer_input = json_["2"]["res"]
        writer_response = json_["3"]["res"]
        runtime = json_["4"]["function_runtime"]
        query = json_["4"]["kwargs"][1:-1].split(": ")[-1].strip("'").strip('"')
    output_filename = "_".join(
        "".join([c for c in query.lower() if c.isalnum() or c == " "]).split(" ")
    )
    output = {
        "data": {
            "query": query,
            "input_to_writer": writer_input,
            "writer_response": writer_response,
            "total_runtime": runtime,
            "log_name": filename,
        },
        "evaluation": {
            "Coverage": "",
            "Coherence": "",
            "Verifiability": "",
            "Validity": "",
            "other_comments": "",
        },
    }
    outputs.append(output)
    with open(f"{OUTPUT_PATH}/{output_filename}.json", "w") as f:
        json.dump(output, f, indent=2)

In [ ]:
print(json.dumps(outputs[0], indent=1))

In [ ]:
shutil.make_archive(OUTPUT_PATH, "gztar", OUTPUT_PATH)